In [1]:
import os, sqlite3
import pandas as pd

PROJECT = os.path.expanduser("~/Documents/olist-analytics")
conn = sqlite3.connect(f"{PROJECT}/olist.db")
os.makedirs(f"{PROJECT}/sql", exist_ok=True)

def q(name, sql):
    """Save the query to the sql/ folder and run it."""
    with open(f"{PROJECT}/sql/{name}.sql", "w") as f:
        f.write(sql.strip() + "\n")
    return pd.read_sql(sql, conn)

In [2]:
q("01_revenue_by_category", """
SELECT p.category,
       COUNT(DISTINCT oi.order_id)            AS orders,
       ROUND(SUM(oi.price), 0)                AS product_revenue,
       ROUND(SUM(oi.price + oi.freight_value), 0) AS total_revenue,
       ROUND(100.0 * SUM(oi.price) / SUM(SUM(oi.price)) OVER (), 1) AS pct_of_revenue
FROM order_items_clean oi
JOIN products_clean p ON oi.product_id = p.product_id
JOIN orders_clean o   ON oi.order_id = o.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY p.category
ORDER BY product_revenue DESC
LIMIT 10
""")

,category,orders,product_revenue,total_revenue,pct_of_revenue
0,health_beauty,8800,1255695.0,1437666.0,9.3
1,watches_gifts,5604,1198185.0,1298292.0,8.9
2,bed_bath_table,9399,1035964.0,1240386.0,7.7
3,sports_leisure,7673,979741.0,1147245.0,7.3
4,computers_accessories,6654,904322.0,1050942.0,6.7
5,furniture_decor,6425,727465.0,899626.0,5.4
6,housewares,5847,626826.0,772035.0,4.6
7,cool_stuff,3616,620770.0,704086.0,4.6
8,auto,3872,586586.0,678607.0,4.3
9,garden_tools,3505,481010.0,579525.0,3.6


In [3]:
q("02_state_orders_and_delays", """
SELECT c.customer_state AS state,
       COUNT(*)                              AS orders,
       ROUND(AVG(o.delivery_days), 1)        AS avg_delivery_days,
       ROUND(AVG(o.delay_days), 1)           AS avg_delay_days,
       ROUND(100.0 * AVG(o.is_late), 1)      AS pct_late
FROM orders_clean o
JOIN customers_clean c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY orders DESC
""")

,state,orders,avg_delivery_days,avg_delay_days,pct_late
0,SP,40501,8.3,-11.1,4.5
1,RJ,12350,14.8,-11.8,12.1
2,MG,11354,11.5,-13.2,4.6
3,RS,5345,14.8,-13.9,6.1
4,PR,4923,11.5,-13.3,4.0
5,SC,3546,14.5,-11.5,8.2
6,BA,3256,18.9,-10.8,12.2
7,DF,2080,12.5,-12.0,5.7
8,ES,1995,15.3,-10.5,10.7
9,GO,1957,15.2,-12.2,6.5


In [4]:
q("03_repeat_customer_rate", """
WITH cust AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS n_orders
    FROM orders_clean o
    JOIN customers_clean c ON o.customer_id = c.customer_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY c.customer_unique_id
)
SELECT COUNT(*)                                             AS customers,
       SUM(CASE WHEN n_orders > 1 THEN 1 ELSE 0 END)        AS repeat_customers,
       ROUND(100.0 * SUM(CASE WHEN n_orders > 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS repeat_rate_pct
FROM cust
""")

,customers,repeat_customers,repeat_rate_pct
0,94990,2888,3.04


In [5]:
q("04_payment_types", """
SELECT payment_type,
       COUNT(*)                                          AS payments,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_payments,
       ROUND(SUM(payment_value), 0)                      AS total_value
FROM order_payments_clean
GROUP BY payment_type
ORDER BY payments DESC
""")

,payment_type,payments,pct_of_payments,total_value
0,credit_card,76795,73.9,12542084.0
1,boleto,19784,19.0,2869361.0
2,voucher,5775,5.6,379437.0
3,debit_card,1529,1.5,217990.0


In [6]:
q("05_reviews_vs_late", """
SELECT CASE WHEN o.is_late = 1 THEN 'Late' ELSE 'On time' END AS delivery,
       COUNT(*)                          AS orders,
       ROUND(AVG(r.review_score), 2)     AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END), 1) AS pct_low_reviews
FROM orders_clean o
JOIN order_reviews_clean r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered' AND o.is_late IS NOT NULL
GROUP BY delivery
""")

,delivery,orders,avg_review_score,pct_low_reviews
0,Late,6381,2.27,62.4
1,On time,89451,4.29,9.3


In [7]:
df = pd.read_sql("""
SELECT o.delay_days, o.is_late, r.review_score
FROM orders_clean o JOIN order_reviews_clean r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered' AND o.delay_days IS NOT NULL
""", conn)
print(df[["delay_days", "is_late", "review_score"]].corr(method="spearman").round(3))

              delay_days  is_late  review_score
delay_days         1.000    0.432        -0.176
is_late            0.432    1.000        -0.311
review_score      -0.176   -0.311         1.000


In [8]:
rfm_src = pd.read_sql("""
SELECT c.customer_unique_id,
       o.order_id,
       o.order_purchase_timestamp AS purchase_ts,
       SUM(oi.price + oi.freight_value) AS order_value
FROM orders_clean o
JOIN customers_clean c ON o.customer_id = c.customer_id
JOIN order_items_clean oi ON o.order_id = oi.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY c.customer_unique_id, o.order_id, o.order_purchase_timestamp
""", conn, parse_dates=["purchase_ts"])

snapshot = rfm_src["purchase_ts"].max() + pd.Timedelta(days=1)

rfm = rfm_src.groupby("customer_unique_id").agg(
    recency=("purchase_ts", lambda s: (snapshot - s.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("order_value", "sum")
).reset_index()

rfm.describe().round(1)

,recency,frequency,monetary
count,94983.0,94983.0,94983.0
mean,243.4,1.0,165.7
std,153.0,0.2,226.7
min,1.0,1.0,9.6
25%,119.0,1.0,63.1
50%,224.0,1.0,107.9
75%,352.0,1.0,182.9
max,729.0,16.0,13664.1


In [9]:
rfm["R"] = pd.qcut(rfm["recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5]).astype(int)
rfm.head()

,customer_unique_id,recency,frequency,monetary,R,F,M
0,0000366f3b9a7992bf8c76cfdf3221e2,116,1,141.90,4,1,4
1,0000b849f77a49e4a4ce2b2a4ca5be3f,119,1,27.19,4,1,1
2,0000f46a3911fa3c0805444483337064,542,1,86.22,1,1,2
3,0000f6ccb0745a6a4b88665a16c9f078,326,1,43.62,2,1,1
4,0004aac84e0df4da2b147fca70cf8255,293,1,196.89,2,1,4


In [10]:
def segment(r):
    if r.R >= 4 and r.F >= 4:                 return "Champions"
    if r.R >= 3 and r.F >= 3:                 return "Loyal"
    if r.R >= 4 and r.F <= 2:                 return "New / Recent"
    if r.R <= 2 and r.F >= 3:                 return "At Risk"
    if r.R <= 2 and r.F <= 2 and r.M >= 4:    return "Can't Lose (big spenders)"
    if r.R <= 2:                              return "Lost"
    return "Needs Attention"

rfm["segment"] = rfm.apply(segment, axis=1)

summary = rfm.groupby("segment").agg(
    customers=("customer_unique_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
    total_revenue=("monetary", "sum")
).round(1).sort_values("customers", ascending=False)
summary

,customers,avg_recency,avg_frequency,avg_monetary,total_revenue
segment,,,,,
At Risk,22663,400.3,1.0,167.7,3800369.1
Loyal,19106,174.0,1.0,161.7,3089187.5
New / Recent,15239,95.9,1.0,163.8,2496629.8
Champions,15221,95.3,1.1,177.6,2702667.1
Lost,9440,401.9,1.0,72.8,687115.8
Needs Attention,7504,225.5,1.0,154.5,1159691.0
Can't Lose (big spenders),5810,400.4,1.0,309.8,1799866.7


In [11]:
rfm.to_csv(f"{PROJECT}/data/clean/rfm_segments.csv", index=False)
summary.to_csv(f"{PROJECT}/data/clean/rfm_summary.csv")

In [12]:
import os
print(os.listdir(f"{PROJECT}/data/clean"))

['order_reviews.csv', 'rfm_summary.csv', 'customers.csv', 'products.csv', 'orders.csv', 'rfm_segments.csv', 'geolocation.csv', 'order_payments.csv', 'sellers.csv', 'order_items.csv']


In [13]:
summary

,customers,avg_recency,avg_frequency,avg_monetary,total_revenue
segment,,,,,
At Risk,22663,400.3,1.0,167.7,3800369.1
Loyal,19106,174.0,1.0,161.7,3089187.5
New / Recent,15239,95.9,1.0,163.8,2496629.8
Champions,15221,95.3,1.1,177.6,2702667.1
Lost,9440,401.9,1.0,72.8,687115.8
Needs Attention,7504,225.5,1.0,154.5,1159691.0
Can't Lose (big spenders),5810,400.4,1.0,309.8,1799866.7


In [14]:
rfm["R"] = pd.qcut(rfm["recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5]).astype(int)
rfm["F"] = rfm["frequency"].clip(upper=3)   # 1 order, 2 orders, 3+ orders
rfm.head()

,customer_unique_id,recency,frequency,monetary,R,F,M,segment
0,0000366f3b9a7992bf8c76cfdf3221e2,116,1,141.90,4,1,4,New / Recent
1,0000b849f77a49e4a4ce2b2a4ca5be3f,119,1,27.19,4,1,1,New / Recent
2,0000f46a3911fa3c0805444483337064,542,1,86.22,1,1,2,Lost
3,0000f6ccb0745a6a4b88665a16c9f078,326,1,43.62,2,1,1,Lost
4,0004aac84e0df4da2b147fca70cf8255,293,1,196.89,2,1,4,Can't Lose (big spenders)


In [15]:
def segment(r):
    if r.F >= 2 and r.R >= 4:  return "Champions (recent repeat buyers)"
    if r.F >= 2:               return "Loyal (repeat, less recent)"
    if r.R >= 4 and r.M >= 4:  return "New high-value"
    if r.R >= 4:               return "New"
    if r.R == 3:               return "Needs attention"
    if r.M >= 4:               return "At risk big spenders"
    return "Lost"

rfm["segment"] = rfm.apply(segment, axis=1)

summary = rfm.groupby("segment").agg(
    customers=("customer_unique_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
    total_revenue=("monetary", "sum")
).round(1).sort_values("customers", ascending=False)
summary

,customers,avg_recency,avg_frequency,avg_monetary,total_revenue
segment,,,,,
Lost,22921,401.6,1.0,72.8,1667790.2
New,22141,94.9,1.0,73.1,1617962.3
Needs attention,18393,225.7,1.0,151.9,2794253.2
New high-value,14676,97.0,1.0,303.0,4447526.2
At risk big spenders,13965,400.2,1.0,309.2,4317779.9
"Loyal (repeat, less recent)",1646,326.5,2.1,298.4,491200.7
Champions (recent repeat buyers),1241,94.0,2.2,321.5,399014.6


In [16]:
rfm.to_csv(f"{PROJECT}/data/clean/rfm_segments.csv", index=False)
summary.to_csv(f"{PROJECT}/data/clean/rfm_summary.csv")

In [17]:
summary


,customers,avg_recency,avg_frequency,avg_monetary,total_revenue
segment,,,,,
Lost,22921,401.6,1.0,72.8,1667790.2
New,22141,94.9,1.0,73.1,1617962.3
Needs attention,18393,225.7,1.0,151.9,2794253.2
New high-value,14676,97.0,1.0,303.0,4447526.2
At risk big spenders,13965,400.2,1.0,309.2,4317779.9
"Loyal (repeat, less recent)",1646,326.5,2.1,298.4,491200.7
Champions (recent repeat buyers),1241,94.0,2.2,321.5,399014.6


In [18]:
rfm.to_csv(f"{PROJECT}/data/clean/rfm_segments.csv", index=False)
summary.to_csv(f"{PROJECT}/data/clean/rfm_summary.csv")